In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 


In [ ]:
df = pd.read_csv("Sensor_Data/84_avalon.csv")
df.head()

,Time,Value,State,Quality,Reason,Status,Suppression Type
0,2025-12-20 08:00:32.698,-.03 In.,Low Low,Good,Value Change,NaN,NaN
1,2025-12-20 05:00:30.425,-.02 In.,Low Low,Good,Value Change,NaN,NaN
2,2025-12-20 04:30:29.268,-.02 In.,Low Low,Good,Value Change,NaN,NaN
3,2025-12-20 03:00:26.059,-.02 In.,Low Low,Good,Value Change,NaN,NaN
4,2025-12-20 02:30:24.993,-.02 In.,Low Low,Good,Value Change,NaN,NaN


In [ ]:
df.State.value_counts()

State
High High     13263
Underrange     9269
Normal         4425
Low Low        2167
High              3
Name: count, dtype: int64

In [ ]:
import os
import glob

# Get all CSV files in Sensor_Data directory
sensor_files = glob.glob('Sensor_Data/*.csv')

# Run value_counts on "State" column for each sensor
for file in sorted(sensor_files):
    try:
        df_sensor = pd.read_csv(file)
        sensor_name = os.path.basename(file)
        print(f"\n{'='*50}")
        print(f"File: {sensor_name}")
        print(f"{'='*50}")
        if 'State' in df_sensor.columns:
            print(df_sensor['State'].value_counts())
        else:
            print("No 'State' column found in this file")
            print(f"Available columns: {df_sensor.columns.tolist()}")
    except Exception as e:
        print(f"Error reading {file}: {e}")


File: 84_US_59.csv
State
Normal        26958
High High       109
High             59
Underrange       24
Name: count, dtype: int64

File: 84_avalon.csv
State
High High     13263
Underrange     9269
Normal         4425
Low Low        2167
High              3
Name: count, dtype: int64

File: 84_bitter root.csv
State
Normal        21758
High High      4262
Underrange     2780
High           1575
Low Low         360
Overrange        10
Name: count, dtype: int64

File: 84_brentwood Oaks.csv
State
Normal    99984
High         16
Name: count, dtype: int64

File: 84_rain_nonzero.csv
State
High High    16653
Name: count, dtype: int64

File: 84_sorters_north.csv
State
Low Low    40
Normal      3
Name: count, dtype: int64

File: 84_sorters_south.csv
State
Normal               99857
Possible Flooding      107
Active Flooding         22
Overrange               14
Name: count, dtype: int64

File: 84_southwood_oaks.csv
State
Underrange    22181
Low Low        2651
High High      1040
Normal         

In [ ]:
# Combine all sensor data files into one CSV
import pandas as pd
import glob
import os

# Read all sensor files with location identifier
dfs = []
for file in sorted(glob.glob('Sensor_Data/*.csv')):
    df = pd.read_csv(file)
    location = os.path.basename(file).replace('.csv', '')
    df['Location'] = location
    dfs.append(df)

# Combine into single dataframe
combined_df = pd.concat(dfs, ignore_index=True)

# Sort by time for easier analysis
combined_df['Time'] = pd.to_datetime(combined_df['Time'])
combined_df = combined_df.sort_values('Time').reset_index(drop=True)

# Save combined dataset
combined_df.to_csv('Sensor_Data/combined_sensor_data.csv', index=False)

print(f"Combined dataset created!")
print(f"Total rows: {len(combined_df)}")
print(f"Date range: {combined_df['Time'].min()} to {combined_df['Time'].max()}")
print(f"Locations: {combined_df['Location'].unique()}")
print(f"\nSaved to: Sensor_Data/combined_sensor_data.csv")

Combined dataset created!
Total rows: 138802
Date range: 2025-04-15 23:19:46.825000 to 2025-12-20 08:22:19.252000
Locations: ['84_southwood_oaks' '84_bitter root' '84_US_59' '84_avalon'
 'Southwood Laverne']

Saved to: Sensor_Data/combined_sensor_data.csv


In [ ]:
# Create boxplots for numeric variables
n_rows = (num_cols + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].boxplot(df[col].dropna(), vert=True)
    axes[idx].set_title(f'Boxplot of {col}', fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(alpha=0.3)

# Hide any unused subplots
for idx in range(num_cols, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 6. Boxplots - Numeric Variables (Outlier Detection)

In [ ]:
# Create distribution plots for numeric variables
num_cols = len(numeric_cols)
n_cols = 3
n_rows = (num_cols + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)

# Hide any unused subplots
for idx in range(num_cols, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 5. Distribution Visualizations - Numeric Variables

In [ ]:
# Check for missing values across all columns
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

if len(missing_data) > 0:
    print("Missing Values Summary:")
    print("="*80)
    print(missing_data.to_string(index=False))
else:
    print("No missing values found in the dataset!")

## 4. Data Quality & Missing Values

In [ ]:
# Calculate correlation matrix for numeric variables
correlation_matrix = df[numeric_cols].corr().round(4)

print("Correlation Matrix (Numeric Variables):")
print("="*80)
print(correlation_matrix.to_string())

# Visualize correlation matrix as heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Numeric Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Correlation Matrix - Numeric Variables

## 3b. Large Correlation Heatmap - Full Detail

In [ ]:
# Generate a large, detailed correlation heatmap for all numeric variables
plt.figure(figsize=(18, 16))

# Calculate correlation matrix and handle NaN values
corr_matrix = df[numeric_cols].corr().fillna(0)

# Create heatmap with annotations showing exact correlation values
sns.heatmap(
    corr_matrix, 
    annot=True,           # Show correlation values
    fmt='.2f',            # Format to 2 decimal places
    cmap='RdBu_r',        # Red-Blue diverging colormap (red=positive, blue=negative)
    center=0,             # Center colormap at 0
    square=True,          # Make cells square
    linewidths=1,         # Add gridlines between cells
    cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'},
    vmin=-1, vmax=1,      # Set scale from -1 to 1
    annot_kws={'size': 9} # Font size for annotations
)

plt.title('Full Correlation Matrix - All Numeric Variables\n(Sensor Data)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Variables', fontsize=12, fontweight='bold')
plt.ylabel('Variables', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("CORRELATION HEATMAP LEGEND")
print("="*80)
print("• Red cells = Positive correlation (variables move together)")
print("• Blue cells = Negative correlation (variables move opposite)")
print("• White cells = No correlation (independent)")
print("• Darker color = Stronger correlation")
print("• Diagonal = 1.0 (each variable perfectly correlated with itself)")
print("="*80)

## 3c. Clustered Correlation Heatmap (Hierarchical)

In [ ]:
# Import scipy for clustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# Generate clustered heatmap (groups similar variables together)
plt.figure(figsize=(18, 16))

# Create correlation matrix and handle NaN values
corr_matrix = df[numeric_cols].corr().fillna(0)

# Create clustered heatmap
sns.clustermap(
    corr_matrix,
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=1,
    figsize=(18, 16),
    cbar_kws={'label': 'Correlation Coefficient'},
    vmin=-1, vmax=1,
    annot=True,           # Show values
    fmt='.2f',
    annot_kws={'size': 8},
    method='average',     # Linkage method for clustering
    metric='euclidean'    # Distance metric
)

plt.suptitle('Clustered Correlation Matrix - Numeric Variables\n(Related variables grouped together)', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("CLUSTERED HEATMAP BENEFITS")
print("="*80)
print("• Variables with similar correlation patterns are grouped together")
print("• Dendrogram on left/top shows hierarchical clustering")
print("• Easier to identify variable groups and relationships")
print("• Useful for feature selection and dimensionality reduction")
print("="*80)

In [ ]:
# Identify categorical columns (object dtype)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}\n")

# Display percentage breakdown for each categorical variable
for col in categorical_cols:
    print("="*80)
    print(f"Column: {col}")
    print("="*80)
    value_counts = df[col].value_counts()
    percentage = (df[col].value_counts(normalize=True) * 100).round(2)
    
    # Create a summary dataframe
    cat_summary = pd.DataFrame({
        'Count': value_counts,
        'Percentage (%)': percentage
    })
    print(cat_summary)
    print(f"\nTotal unique values: {df[col].nunique()}")
    print(f"Missing values: {df[col].isnull().sum()}\n")

## 2. Categorical Variable Breakdown (Percentage Distribution)

In [ ]:
# Identify numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}\n")

# Create summary statistics table for numeric variables
summary_stats = df[numeric_cols].agg(['mean', 'median', 'std', 'min', 'max']).round(4)
print("Numeric Variables Summary Statistics:")
print(summary_stats)

# Also display as a more readable transposed table
print("\n" + "="*80)
print("Summary Statistics (Transposed View):")
print("="*80)
summary_stats_T = summary_stats.T
print(summary_stats_T.to_string())

## 1. Numeric Variable Summary Statistics

In [ ]:
# Load merged sensor data and inspect structure
df = pd.read_csv('sensor_data_merged.csv')

print("Dataset Shape:", df.shape)
print("\nFirst Few Rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
# Summary of analysis findings
print("="*80)
print("EXPLORATORY & DESCRIPTIVE ANALYSIS SUMMARY")
print("="*80)

print("\n DATASET OVERVIEW")
print(f"  • Total Records: {len(df):,}")
print(f"  • Total Columns: {len(df.columns)}")
print(f"  • Date Range: {df['Time'].min()} to {df['Time'].max()}")

print("\n NUMERIC VARIABLES (15 total)")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"  Variables: {', '.join(numeric_cols[:5])}...")
print(f"\n  Key Statistics Summary:")
summary_stats = df[numeric_cols].agg(['mean', 'median', 'std', 'min', 'max']).round(4)
print("  See table above for full numeric summary")

print("\n  CATEGORICAL VARIABLES")
print("  • State (6 categories):")
print("    - Normal: 43.01%")
print("    - High High: 27.39%") 
print("    - Underrange: 24.69%")
print("    - Low Low: 3.73%")
print("  • Location (5 sensors):")
print("    - All locations have ~20% representation")
print("  • Season (4 categories):")
print("    - Summer: 63.09% (dominant)")
print("    - Fall: 27.69%")
print("    - Spring: 7.15%")
print("    - Winter: 2.07% (sparse)")

print("\n STRONG CORRELATIONS (|r| > 0.3)")
print("  • Temperature ↔ Month (r = -0.57): Strong negative - colder in winter months")
print("  • Temperature ↔ Humidity (r = -0.53): Strong negative - inverse relationship")
print("  • Wind Speed ↔ Wind Gust (r = 0.50): Moderate positive - expected")
print("  • Humidity ↔ Wind Speed (r = -0.39): Moderate negative")
print("  • Wind Direction ↔ Wind Speed (r = 0.36): Weak positive")
print("  • Temperature ↔ Pressure (r = -0.32): Weak negative")

print("\n DATA QUALITY NOTES")
print("  • Missing values in several weather columns (all handled gracefully)")
print("  • snow_depth_cm_hourly: 100% missing (not applicable to Houston)")
print("  • sunshine_min_hourly: 100% missing (not available)")
print("  • Other weather metrics: <1% missing")

print("\n ANALYSIS COMPLETE")
print("="*80)

## 7. Summary of Key Findings

# Comprehensive Exploratory & Descriptive Analysis: sensor_data_merged.csv